In [ ]:
import importlib
import sys
import numpy as np
from collections import Counter
import pandas as pd

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [ ]:
# Load the data
# Path to your pickle file (saved with torch.save)
file_path_train = '../../../../../../data/Helpdesk/tensor_data/normal/helpdesk_all_5_train.pkl'
# Load the dataset using torch.load
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)

# Path to your pickle file (saved with torch.save)
file_path_val = '../../../../../../data/Helpdesk/tensor_data/normal/helpdesk_all_5_val.pkl'
# Load the dataset using torch.load
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)

In [ ]:
# Helpdesk dataset dynamic categories, features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
# print(helpdesk_all_categories_cat)
helpdesk_all_categories_num = helpdesk_all_categories[1]
# print(helpdesk_all_categories_num)
for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")
print('\n')
     
# Helpdesk dataset static categories, features:
helpdesk_all_stat_categories = helpdesk_train_dataset.all_static_categories

helpdesk_all_stat_categories_cat = helpdesk_all_stat_categories[0]
# print(helpdesk_all_stat_categories_cat)
helpdesk_all_stat_categories_num = helpdesk_all_stat_categories[1]
# print(helpdesk_all_stat_categories_num)
for i, cat in enumerate(helpdesk_all_stat_categories_cat):
     print(f"Helpdesk static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_stat_categories_num):
     print(f"Helpdesk static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")  

In [ ]:
# Create lists with name of Encoder features (input) and decoder features (input & output)
# Encoder features:
enc_feat_cat = []
enc_feat_num = []
for cat in helpdesk_all_categories_cat:
    enc_feat_cat.append(cat[0])
for num in helpdesk_all_categories_num:
    enc_feat_num.append(num[0])
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Static encoder features:
stat_enc_feat_cat = []
stat_enc_feat_num = []
for cat in helpdesk_all_stat_categories_cat:
     stat_enc_feat_cat.append(cat[0])
for num in helpdesk_all_stat_categories_num:
     stat_enc_feat_num.append(num[0])
stat_enc_feat = [stat_enc_feat_cat, stat_enc_feat_num]
print("Input features static encoder: ", stat_enc_feat)

# Decoder features:
dec_feat_cat = ['Activity']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

In [ ]:
import suffix_pred.models.K_UED_LSTM
importlib.reload(suffix_pred.models.K_UED_LSTM)
from suffix_pred.models.K_UED_LSTM import DropoutUncertaintyEncoderDecoderLSTM

# Prediction decoder output sequence length
seq_len_pred = helpdesk_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 128

# Number of cells
num_layers = 4

# Fixed Dropout probability 
# Experiment with different values but be carefull with exploiting gradients:
dropout = 0.1

# Encoder Decoder model initialization
model = DropoutUncertaintyEncoderDecoderLSTM(data_set_categories=helpdesk_all_categories,
                                             enc_feat=enc_feat,
                                             dec_feat=dec_feat,
                                             seq_len_pred=seq_len_pred,
                                             hidden_size=hidden_size,
                                             num_layers=num_layers,
                                             dropout=dropout,
                                             # static attributes
                                             static_data_set_categories=helpdesk_all_stat_categories,
                                             static_enc_feat=stat_enc_feat
                                             )

In [ ]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [ ]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import KTrainer

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)

# Epochs / Batch size
num_epochs = 100
batch_size = 128

# L2 regularization term
regularization_term = 1e-4

# Shuffle data
shuffle = True

# Teacher forcing schedule
max_teacher_forcing_value = 1.0
min_teacher_forcing_value = 0.0

optimize_values = {
    "regularization_term": regularization_term,
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
    "min_teacher_forcing_value": min_teacher_forcing_value,
    "max_teacher_forcing_value": max_teacher_forcing_value,
}

required_optimize_keys = {
    "regularization_term",
    "optimizer",
    "scheduler",
    "epochs",
    "mini_batches",
    "shuffle",
    "min_teacher_forcing_value",
    "max_teacher_forcing_value",
}
missing_keys = required_optimize_keys.difference(optimize_values.keys())
if missing_keys:
    raise ValueError(f"Missing required optimize_values keys: {sorted(missing_keys)}")

suffix_data_split_value = helpdesk_train_dataset.min_suffix_size
print("Train seq length:", suffix_data_split_value)

import os
model_output_path = "../../../../../../models/Helpdesk/clean/Helpdesk_UED_LSTM_v1_clean.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = KTrainer(
    device=device,
    model=model,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    loss_obj=loss_obj,
    log_normal_loss_num_feature=[],
    optimize_values=optimize_values,
    suffix_data_split_value=suffix_data_split_value,
    save_model_n_th_epoch=1,
    saving_path=model_output_path,
)

# Train the model
train_attenuated_losses, val_losses, val_attenuated_losses = trainer.train_model(use_statics=True,
                                                                                 use_zero_padd_masking=True,
                                                                                 use_eos_padd_masking=True)
